# TFM Deteccion - Ejecucion final

Pipeline definitivo: solo CNNDetection, 3 modelos (ResNet-50, ViT-B/16, UFD; FreqNet en stand-by), trainset de ejecucion 100.000 imgs (5000 por categoria, balance 50/50 real/fake) sobre las 20 categorias de progan_train. Validacion (progan_val) y test (progan_test + CNN_synth_testset, E1+E1b) completos.

El trainset se construye via API de Drive con streaming + range requests (NO via el mount FUSE) para evitar que Colab descargue el zip entero como cache local. Ver `problema_carga_datasets.md` para detalles.

Requisitos:
- Runtime Colab con GPU (T4 o superior).
- En `MyDrive/cnndetection-datasets/` (o como acceso directo): `progan_train.zip`, `progan_val.zip`, `progan_testset.zip`, `CNN_synth_testset.zip`.
- (Opcional) cuenta de wandb.


## 1. Drive + repo + dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!rm -rf /content/tfm_deteccion
!git clone https://github.com/manuelpalasanchez/tfm_deteccion.git
%cd /content/tfm_deteccion
!git pull

In [ ]:
!pip install -q -r requirements.txt

## 2. Inventario del progan_train.zip (opcional)

Lee el indice del zip directamente desde Drive (sin descomprimir nada). Ya hecho una vez: 20 cats x 36k imgs (50/50). Ejecutar solo si quieres reverificar.

In [ ]:
ZIP_TRAIN = '/content/drive/MyDrive/cnndetection-datasets/progan_train.zip'
!python scripts/scan_cnndetection.py --zip {ZIP_TRAIN} --csv reports/scan_progan_train.csv

## 3. Construccion del trainset de ejecucion (via Drive API)

Para evitar el problema de la cache FUSE de Drive (que descarga ~3.5 GB del zip por categoria accedida y llena el disco de Colab), usamos la API REST de Drive con range requests + streaming. Solo se transfiere el contenido de los ficheros seleccionados (~7-10 GB para N=5000), no el zip entero.

Las 20 categorias estan balanceadas (~36k imgs cada una), asi que muestreamos N fijo por categoria con balance 50/50 real-fake. Misma particion (seed=42) para los 3 modelos.


In [ ]:
# Autenticar con Drive API (no es lo mismo que drive.mount; este flujo no usa FUSE)
from google.colab import auth as colab_auth
colab_auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build

creds, _ = google.auth.default()
service = build('drive', 'v3', credentials=creds)

# Buscar el progan_train.zip (puede ser un acceso directo a otro Drive)
res = service.files().list(
    q="name='progan_train.zip' and trashed=false",
    fields="files(id,name,mimeType,size,shortcutDetails,owners)",
    supportsAllDrives=True,
    includeItemsFromAllDrives=True,
).execute()

matches = res.get('files', [])
if not matches:
    raise SystemExit('No se encontro ningun progan_train.zip accesible.')

print('Coincidencias encontradas:')
for f in matches:
    is_shortcut = f.get('mimeType') == 'application/vnd.google-apps.shortcut'
    target = f.get('shortcutDetails', {}).get('targetId') if is_shortcut else None
    size = f.get('size', '?')
    print('  id=' + f['id'] + '  shortcut=' + str(is_shortcut) + '  target=' + str(target) + '  size=' + str(size) + '  name=' + f['name'])

FILE_ID = matches[0]['id']
print('')
print('Usando FILE_ID = ' + FILE_ID)

N_PER_CAT = 5000
SEED = 42
OUT_TRAIN = '/content/cnndetection/progan_train'

!python scripts/build_trainset_drive_api.py --file-id {FILE_ID} --out {OUT_TRAIN} --n-per-cat {N_PER_CAT} --seed {SEED} --manifest reports/trainset_ejecucion_manifest.json


## 4. Extraccion de val + tests (completos)

progan_val (validacion durante el entrenamiento), progan_testset (E1) y CNN_synth_testset (E1b).

In [ ]:
import os, subprocess

DRIVE_ZIPS = '/content/drive/MyDrive/cnndetection-datasets'
CNN_ROOT = '/content/cnndetection'

for zip_name, dst in [
    ('progan_val.zip',        f'{CNN_ROOT}/progan_val'),
    ('progan_testset.zip',    CNN_ROOT),
    ('CNN_synth_testset.zip', CNN_ROOT),
]:
    print(f'extrayendo {zip_name}...')
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['unzip', '-q', '-n', f'{DRIVE_ZIPS}/{zip_name}', '-d', dst], check=True)

if os.path.isdir(f'{CNN_ROOT}/progan_testset') and not os.path.isdir(f'{CNN_ROOT}/progan_test'):
    os.rename(f'{CNN_ROOT}/progan_testset', f'{CNN_ROOT}/progan_test')
    print('renombrado progan_testset -> progan_test')

print('\nEstructura:')
!ls {CNN_ROOT}
!df -h /content | grep -v Filesystem

## 5. Patch de base.yaml

Apunta los roots a `/content/cnndetection`, habilita E1+E1b, deja E2 desactivado y guarda checkpoints en Drive para sobrevivir caidas de sesion.

In [ ]:
import yaml, pathlib

cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

cfg['data']['train']['root']       = '/content/cnndetection'
cfg['data']['val']['root']         = '/content/cnndetection'
cfg['data']['eval_e1']['root']     = '/content/cnndetection'
cfg['data']['eval_e1']['split']    = 'test'
cfg['data']['eval_e1b']['root']    = '/content/cnndetection'
cfg['data']['eval_e1b']['enabled'] = True
cfg['data']['eval_e2']['enabled']  = False
cfg['data']['num_workers']         = 2

cfg['training']['epochs']     = 8
cfg['training']['batch_size'] = 128
cfg['training']['scheduler']['T_max'] = 8

cfg['output']['base_dir'] = '/content/drive/MyDrive/tfm-checkpoints'
cfg['wandb']['enabled'] = True

cfg_path.write_text(yaml.dump(cfg))
print(yaml.dump(cfg))

## 6. wandb (opcional)

Si no quieres tracking, salta esta celda y pon `cfg['wandb']['enabled'] = False` arriba.

In [ ]:
!wandb login

## 7bis. Fix de nombres post-extraccion (obligatorio)

El zip de progan_testset extrajo a una carpeta `progan/` y CNN_synth_testset extrajo todos los archs sueltos en `/content/cnndetection/`. Renombrar `progan -> progan_test` y mover los 12 archs a un root separado para que E1 (in-dist) y E1b (cross-arch) carguen datasets distintos.


In [ ]:
!mkdir -p /content/CNN_synth_testset
!mv /content/cnndetection/progan /content/cnndetection/progan_test 2>/dev/null || echo 'progan ya renombrado'
!for d in biggan crn cyclegan deepfake gaugan imle san seeingdark stargan stylegan stylegan2 whichfaceisreal; do mv /content/cnndetection/$d /content/CNN_synth_testset/${d}_test 2>/dev/null; done

import yaml, pathlib
p = pathlib.Path('configs/base.yaml')
c = yaml.safe_load(p.read_text())
c['data']['eval_e1b']['root'] = '/content/CNN_synth_testset'
p.write_text(yaml.dump(c))

print('--- /content/cnndetection ---')
!ls /content/cnndetection
print('--- /content/CNN_synth_testset ---')
!ls /content/CNN_synth_testset


## 7. Sanity check

In [ ]:
import sys
sys.path.insert(0, '/content/tfm_deteccion')
from data.cnndetection_dataset import CNNDetectionDataset
from data.transforms import get_eval_transforms

for split in ('train', 'val', 'test'):
    try:
        ds = CNNDetectionDataset(root='/content/cnndetection', split=split, transform=get_eval_transforms())
        print(f'{split}: {len(ds)} muestras, generadores={ds.generators}')
    except Exception as e:
        print(f'{split}: ERROR - {e}')

# Cross-arch (E1b) lo carga el evaluator desde /content/CNN_synth_testset
try:
    ds_e1b = CNNDetectionDataset(root='/content/CNN_synth_testset', split='test', transform=get_eval_transforms())
    print(f'e1b: {len(ds_e1b)} muestras, generadores={ds_e1b.generators}')
except Exception as e:
    print(f'e1b: ERROR - {e}')


## 8. Train + evaluate (modelo a modelo)

Tres bloques: ResNet-50, ViT-B/16, UniversalFakeDetect. Cada bloque tiene una celda de train y otra de eval. Los checkpoints se guardan en `/content/drive/MyDrive/tfm-checkpoints/{modelo}_<timestamp>/` y la celda de eval localiza automaticamente el ultimo run.

**Recuperacion**: si un modelo ya esta entrenado en una sesion anterior (su carpeta tiene `checkpoint_best.pth`), saltar la celda de train y correr solo la de eval. Si la carpeta tambien tiene `metrics.json` y los PNGs, ese modelo esta completo y se puede saltar entero.

Configs ya parcheados:
- `configs/vit.yaml`: `batch_size=64` (ViT-B/16 no entra a 128 en T4).
- `configs/universalfakedetect.yaml`: `epochs=3` (CLIP frozen, la cabeza converge rapido y 8 epocas no caben en una sesion).


### ResNet-50


In [ ]:
# Train ResNet-50 (~50 min en T4)
!python scripts/train.py --config configs/resnet50.yaml


In [ ]:
# Eval ResNet-50: localiza el ultimo run y corre evaluate.py (~5 min)
import glob
modelo = 'resnet50'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"


### ViT-B/16


In [ ]:
# Train ViT-B/16 (~2-2.5 h en T4 con batch=64)
!python scripts/train.py --config configs/vit.yaml


In [ ]:
# Eval ViT-B/16
import glob
modelo = 'vit'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"


### UniversalFakeDetect


In [ ]:
# Train UFD (~3-4 h en T4; con CLIP frozen + 3 epocas configuradas)
!python scripts/train.py --config configs/universalfakedetect.yaml


In [ ]:
# Eval UFD
import glob
modelo = 'universalfakedetect'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"


## 9. Resumen de metricas

In [ ]:
import glob, json, csv, os

CKPT_BASE = '/content/drive/MyDrive/tfm-checkpoints'
modelos = ['resnet50', 'vit', 'universalfakedetect']
rows = []

for modelo in modelos:
    dirs = sorted(glob.glob(f'{CKPT_BASE}/{modelo}_*/metrics.json'))
    if not dirs:
        print(f'{modelo}: sin metrics.json todavia')
        continue
    metrics_path = dirs[-1]
    with open(metrics_path) as f:
        m = json.load(f)
    row = {'modelo': modelo}
    for ronda, vals in m.items():
        row[f'{ronda}_auc']  = round(vals.get('auc_roc', float('nan')), 4)
        row[f'{ronda}_ap']   = round(vals.get('average_precision', float('nan')), 4)
        row[f'{ronda}_acc']  = round(vals.get('accuracy', float('nan')), 4)
    rows.append(row)
    print(f'{modelo}: {metrics_path}')
    for k, v in row.items():
        if k != 'modelo':
            print(f'  {k}: {v}')

if rows:
    csv_path = f'{CKPT_BASE}/resumen_metricas.csv'
    fieldnames = list(rows[0].keys())
    with open(csv_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f'\nCSV guardado en {csv_path}')
